In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

#load dataset and variables
df = pd.read_csv("../data/Basketball_Training_Movement_Biomechanics_Dataset.csv")
X = df[[
    "movement_type", "hip_flexion_angle_deg", "hip_abduction_angle_deg", 
    "knee_flexion_angle_deg", "ankle_flexion_angle_deg", "shoulder_flexion_angle_deg", 
    "shoulder_abduction_angle_deg", "elbow_flexion_angle_deg", "wrist_flexion_angle_deg", 
    "trunk_flexion_angle_deg", "hip_angular_velocity_deg_s", "knee_angular_velocity_deg_s", 
    "ankle_angular_velocity_deg_s", "shoulder_angular_acceleration_deg_s2", 
    "elbow_angular_acceleration_deg_s2", "wrist_angular_acceleration_deg_s2", 
    "center_of_mass_velocity_m_s", "vertical_jump_height_cm", "ground_contact_time_ms", 
    "movement_duration_ms", "balance_score", "movement_stability_score", 
    "movement_symmetry_pct"
]]
y = df["movement_efficiency_score"]

In [3]:
#split data by groups
#using player_id as the group because don't want the model to learn from that
groups = df["player_id"]

#split: training - 70%, validation - 15%, testing - 15%
#outer split for training and validation/testing
outer = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=42)
train_idx, temp_idx = next(outer.split(X, y, groups=groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_temp, y_temp = X.iloc[temp_idx], y.iloc[temp_idx]
groups_temp = groups.iloc[temp_idx]

#inner split to divide validation and testing, still don't want to group by player's id
groups_temp = groups.iloc[temp_idx]
inner = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
val_idx, test_idx = next(inner.split(X, y, groups=groups_temp))

X_val, y_val = X_temp.iloc[val_idx], y_temp.iloc[val_idx]
X_test, y_test = X_temp.iloc[test_idx], y_temp.iloc[test_idx]

print(f"Training: {X_train.shape[0]} rows | {len(set(groups.iloc[train_idx]))} unique players")
print(f"Validation: {X_val.shape[0]} rows | {len(set(groups_temp.iloc[val_idx]))} unique players")
print(f"Testing: {X_test.shape[0]} rows | {len(set(groups_temp.iloc[test_idx]))} unique players")

ValueError: Found input variables with inconsistent numbers of samples: [15000, 15000, 4339]